# Waabi jobs classification prototype

This notebook follows the Waymo classification workflow while using the
normalized output from `scrapers/waabi_scraper.py`. It demonstrates:

1. loading and validating Waabi job data;
2. selecting AV-relevant roles;
3. extracting a small, traceable role taxonomy with an LLM;
4. extracting salary ranges only when they appear in a job advertisement;
5. combining classification and salary results for later analysis.

The notebook does not invent missing values. Every result retains its
source job ID and URL.


## 1. Load the normalized Waabi dataset

Run `python scrapers/waabi_scraper.py` from the repository root before
running this notebook. The scraper creates `data/waabi_jobs.json`.


In [1]:
# Install project dependencies once from the repository root:
# pip install -r requirements.txt


In [2]:
import html
import json
import os
import re
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from groq import Groq


def find_project_root() -> Path:
    """Find the repository whether Jupyter starts in root or notebooks/."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "waabi_jobs.json").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find data/waabi_jobs.json. Run "
        "'python scrapers/waabi_scraper.py' first."
    )


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "waabi_jobs.json"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"

# Support either a repository-root .env or notebooks/.env.
load_dotenv(PROJECT_ROOT / ".env")
load_dotenv(NOTEBOOK_DIR / ".env", override=False)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input data:   {DATA_PATH}")


Project root: C:\Users\Harshil\Documents\projects\Autonomous_Vehicle_Job_Profiles_Group3
Input data:   C:\Users\Harshil\Documents\projects\Autonomous_Vehicle_Job_Profiles_Group3\data\waabi_jobs.json


In [3]:
with DATA_PATH.open("r", encoding="utf-8") as file:
    job_records = json.load(file)

if not isinstance(job_records, list) or not job_records:
    raise ValueError("Waabi JSON must contain a non-empty list of jobs.")

jobs_df = pd.DataFrame(job_records)
required_columns = {
    "company",
    "job_id",
    "job_title",
    "location",
    "description",
    "source_url",
    "posting_date",
}
missing_columns = sorted(required_columns - set(jobs_df.columns))
if missing_columns:
    raise ValueError(f"Waabi data is missing columns: {missing_columns}")

jobs_df["job_id"] = jobs_df["job_id"].astype(str)
duplicate_count = int(jobs_df["job_id"].duplicated().sum())
if duplicate_count:
    raise ValueError(f"Waabi data contains {duplicate_count} duplicate job IDs.")

print(f"Loaded jobs: {len(jobs_df)}")
print(f"Columns: {len(jobs_df.columns)}")
print(f"Duplicate job IDs: {duplicate_count}")


Loaded jobs: 57
Columns: 17
Duplicate job IDs: 0


In [4]:
quality_summary = pd.Series(
    {
        "total_jobs": len(jobs_df),
        "unique_job_ids": jobs_df["job_id"].nunique(),
        "jobs_with_descriptions": jobs_df["description"].ne("").sum(),
        "jobs_with_posting_dates": jobs_df["posting_date"].notna().sum(),
        "distinct_locations": jobs_df["location"].nunique(),
        "distinct_teams": jobs_df["team"].nunique(),
    },
    name="value",
)
quality_summary.to_frame()


,value
total_jobs,57
unique_job_ids,57
jobs_with_descriptions,57
jobs_with_posting_dates,57
distinct_locations,7
distinct_teams,16


In [5]:
display_columns = [
    "job_id",
    "job_title",
    "location",
    "team",
    "commitment",
    "workplace_type",
    "posting_date",
    "source_url",
]
jobs_df[display_columns].head(10)


,job_id,job_title,location,team,commitment,workplace_type,posting_date,source_url
0,62700386-b9db-4c78-aec3-5ef59cbe841e,"2026 Intern, PhD Research Scientist","Toronto, ON",Research,Intern,onsite,2025-12-18T21:59:13.993000Z,https://jobs.lever.co/waabi/62700386-b9db-4c78...
1,df57aa7f-9ce4-46f7-93a5-f8e06c3ae0f3,Applied Scientist,"Toronto, ON",Autonomy & Algorithms,Full-time,hybrid,2026-05-12T04:28:23.992000Z,https://jobs.lever.co/waabi/df57aa7f-9ce4-46f7...
2,a6814f7d-d06d-47cf-bd6b-23dc2d864fad,"Director, OEM & Strategic Partnerships","San Francisco, CA",Business Development & Strategy,Full-time,hybrid,2025-12-04T15:17:46.133000Z,https://jobs.lever.co/waabi/a6814f7d-d06d-47cf...
3,31f542cf-0357-45b9-9d64-0792469d78e1,Distillation Lead,"San Francisco, CA",Autonomy & Algorithms,Full-time,hybrid,2026-04-30T13:35:46.839000Z,https://jobs.lever.co/waabi/31f542cf-0357-45b9...
4,2953d85f-fd51-4fd9-9a44-7517cb0f6e26,Electrical Engineering Team Lead,"Pittsburgh, PA",Embedded Hardware Engineering,Full-time,onsite,2026-08-04T19:26:57.896000Z,https://jobs.lever.co/waabi/2953d85f-fd51-4fd9...
5,71546dca-a904-4620-8e8f-190a3954b6dd,Fleet Specialist,"Dallas, TX",Road Operations,Full-time,onsite,2025-01-24T01:20:12.452000Z,https://jobs.lever.co/waabi/71546dca-a904-4620...
6,ac31b674-bcdf-4666-beef-562bae464c4e,HR Business Partner,"Dallas, TX",Human Resources,Full-time,onsite,2026-04-01T01:37:45.534000Z,https://jobs.lever.co/waabi/ac31b674-bcdf-4666...
7,571033e1-509b-4da2-9f38-76d76b34d234,IT Systems Administrator,"Pittsburgh, PA",Corporate IT,Full-time,onsite,2026-08-04T16:43:43.912000Z,https://jobs.lever.co/waabi/571033e1-509b-4da2...
8,7b8669d5-c11e-49a3-8cfd-492484999d6d,"Lead Product Manager, Autonomy","San Francisco, CA",Product & Technical Program Mgmt,Full-time,hybrid,2026-04-08T17:03:42.483000Z,https://jobs.lever.co/waabi/7b8669d5-c11e-49a3...
9,fc6553a1-eff7-4f79-96fb-1cf97dcb866d,"Lead Technical Program Manager, Platform Integ...","Toronto, ON",Product & Technical Program Mgmt,Full-time,hybrid,2026-07-14T16:27:29.945000Z,https://jobs.lever.co/waabi/fc6553a1-eff7-4f79...


## 2. Select roles and define the classification workflow

The title filter is an initial research aid, not the final category. The
LLM receives the complete job description and returns a concise role
profile, specific technologies, and a broad functional area.


In [6]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scrapers.service.job_prefilter import JobPrefilter

prefilter = JobPrefilter.from_config(
    PROJECT_ROOT / "notebooks" / "config" / "job_prefilter.yaml"
)
prefilter_result = prefilter.filter(jobs_df.to_dict(orient="records"))
prefilter_result.write_outputs(PROJECT_ROOT / "data" / "job_prefilter" / "waabi")
included_job_ids = {
    decision.job_id for decision in prefilter_result.decisions if decision.included
}
technical_jobs_df = jobs_df[
    jobs_df["job_id"].astype(str).isin(included_job_ids)
].copy()

print(f"All Waabi jobs: {prefilter_result.before_count}")
print(f"Jobs sent to the LLM: {prefilter_result.after_count}")
print(f"Jobs retained in the exclusion audit: {len(prefilter_result.excluded)}")
pd.DataFrame(prefilter_result.company_metrics)


All Waabi jobs: 57
Technical/AV title matches: 29


,job_id,job_title,team,location
7,571033e1-509b-4da2-9f38-76d76b34d234,IT Systems Administrator,Corporate IT,"Pittsburgh, PA"
8,7b8669d5-c11e-49a3-8cfd-492484999d6d,"Lead Product Manager, Autonomy",Product & Technical Program Mgmt,"San Francisco, CA"
9,fc6553a1-eff7-4f79-96fb-1cf97dcb866d,"Lead Technical Program Manager, Platform Integ...",Product & Technical Program Mgmt,"Toronto, ON"
11,933f2e35-ffaf-41db-b836-56de0323f519,Physical Infrastructure Engineer (On-premise),Software Engineering,"Dallas, TX"
12,5fcde68c-577a-4fed-b532-793fec8de175,Platform Systems Engineer; Motion Planning and...,Systems Engineering,"Phoenix, AZ"
13,d6e94165-fb89-4dee-b83a-01310cbf8464,Platform Systems Engineer; Sensing and Percept...,Systems Engineering,"Phoenix, AZ"
14,d0748dc8-f1c0-4f9b-9cbd-0c8b4e22ca7d,Platform Verification Engineer - Embedded Systems,Systems Engineering,"San Francisco, CA"
15,c45156f5-792e-459e-a312-435e66728733,Platform Verification Engineer - Track Testing,Systems Engineering,"Phoenix, AZ"
24,48a13158-5489-4fac-8643-2d4e2fdc410e,"Research Scientist, Simulation Agents",Autonomy & Algorithms,Remote US & Canada
27,562aa158-f9c9-4ca4-bfaa-50a62d7e498c,Senior / Staff Graphics Software Engineer,Autonomy & Algorithms,"Toronto, ON"


In [7]:
groq_api_key = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")
groq_client = Groq(api_key=groq_api_key) if groq_api_key else None

if groq_client is None:
    print(
        "GROQ_API_KEY is not set. Data and salary cells will still run; "
        "LLM classification will be skipped."
    )
else:
    print(f"Groq client ready. Model: {GROQ_MODEL}")


GROQ_API_KEY is not set. Data and salary cells will still run; LLM classification will be skipped.


In [8]:
job_prompt = """You are analyzing one job posting from Waabi, an
autonomous-vehicle technology company. Use ONLY the supplied job data.

Extract:
1. A short role_profile name specific to this job. Do not force
   non-engineering roles into an AV engineering category.
2. Specific technical skills, tools, platforms, and technologies that
   are explicitly mentioned. Do not infer unmentioned skills.
3. One broad functional_area, such as Onboard / Vehicle Software,
   Off-board / Cloud Platform, Simulation, Hardware, Safety / Systems,
   Business / Product, Operations, or Corporate / Support.

Respond ONLY with valid JSON using exactly this shape:
{
  "company": "Waabi",
  "title": "...",
  "career_page_url": "...",
  "role_profile": "...",
  "skills": ["..."],
  "functional_area": "..."
}
"""


In [9]:
CLASSIFICATION_FIELDS = {
    "company",
    "title",
    "career_page_url",
    "role_profile",
    "skills",
    "functional_area",
}


def parse_llm_json(content: str) -> dict:
    text = (content or "").strip()
    if text.startswith("```"):
        text = text.strip("`").strip()
        if text.lower().startswith("json"):
            text = text[4:].strip()

    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end <= start:
            raise
        parsed = json.loads(text[start : end + 1])

    if not isinstance(parsed, dict):
        raise ValueError("LLM response must be a JSON object.")
    missing = sorted(CLASSIFICATION_FIELDS - set(parsed))
    if missing:
        raise ValueError(f"LLM response is missing fields: {missing}")
    if not isinstance(parsed["skills"], list):
        raise ValueError("LLM skills must be a JSON list.")
    return parsed


def classify_job(job: pd.Series) -> dict:
    if groq_client is None:
        raise RuntimeError("Set GROQ_API_KEY before classifying jobs.")

    context = {
        "job_id": job["job_id"],
        "title": job["job_title"],
        "team": job.get("team", ""),
        "location": job["location"],
        "career_page_url": job["source_url"],
        "description": job["description"],
    }
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "user",
                "content": job_prompt + "\n\nJOB DATA:\n" + json.dumps(context),
            }
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )
    parsed = parse_llm_json(response.choices[0].message.content)

    # Preserve source-of-truth identifiers instead of trusting model copies.
    parsed["job_id"] = str(job["job_id"])
    parsed["company"] = "Waabi"
    parsed["title"] = job["job_title"]
    parsed["career_page_url"] = job["source_url"]
    return parsed


In [10]:
SAMPLE_SIZE = 3
sample_source_df = technical_jobs_df
sample_jobs_df = sample_source_df.head(SAMPLE_SIZE).copy()

print(f"Selected {len(sample_jobs_df)} sample jobs dynamically:")
sample_jobs_df[["job_id", "job_title", "location", "source_url"]]


Selected 3 sample jobs dynamically:


,job_id,job_title,location,source_url
7,571033e1-509b-4da2-9f38-76d76b34d234,IT Systems Administrator,"Pittsburgh, PA",https://jobs.lever.co/waabi/571033e1-509b-4da2...
8,7b8669d5-c11e-49a3-8cfd-492484999d6d,"Lead Product Manager, Autonomy","San Francisco, CA",https://jobs.lever.co/waabi/7b8669d5-c11e-49a3...
9,fc6553a1-eff7-4f79-96fb-1cf97dcb866d,"Lead Technical Program Manager, Platform Integ...","Toronto, ON",https://jobs.lever.co/waabi/fc6553a1-eff7-4f79...


In [11]:
classification_records = []

if groq_client is None:
    print("Classification skipped. Add GROQ_API_KEY to .env and rerun this cell.")
else:
    for _, job in sample_jobs_df.iterrows():
        try:
            classification_records.append(classify_job(job))
            print(f"Classified: {job['job_title']}")
        except Exception as exc:
            print(f"Classification failed for {job['job_id']}: {exc}")
        time.sleep(0.5)

classification_columns = [
    "job_id",
    "company",
    "title",
    "career_page_url",
    "role_profile",
    "skills",
    "functional_area",
]
classification_df = pd.DataFrame(
    classification_records,
    columns=classification_columns,
)
classification_df


Classification skipped. Add GROQ_API_KEY to .env and rerun this cell.


,job_id,company,title,career_page_url,role_profile,skills,functional_area


## 3. Extract salary ranges from Waabi job descriptions

Salary values are extracted only from the original job advertisement.
Missing compensation remains missing; no third-party estimate or invented
value is added.


In [12]:
SALARY_RANGE_PATTERN = re.compile(
    r"(?P<symbol>[$£€])\s*"
    r"(?P<minimum>\d{2,3}(?:,\d{3})+(?:\.\d{1,2})?)\s*"
    r"(?:-|–|—|to)\s*"
    r"(?:[$£€]\s*)?"
    r"(?P<maximum>\d{2,3}(?:,\d{3})+(?:\.\d{1,2})?)"
    r"(?:\s*(?P<currency>USD|CAD|GBP|EUR))?",
    flags=re.IGNORECASE,
)
SYMBOL_TO_CURRENCY = {"$": "USD", "£": "GBP", "€": "EUR"}


def extract_salary(description: str) -> pd.Series:
    text = html.unescape(description) if isinstance(description, str) else ""
    match = SALARY_RANGE_PATTERN.search(text)
    if not match:
        return pd.Series(
            {
                "salary_min": None,
                "salary_max": None,
                "salary_currency": None,
                "salary_period": None,
                "salary_snippet": None,
                "salary_source": None,
            }
        )

    snippet_start = max(0, match.start() - 60)
    snippet_end = min(len(text), match.end() + 80)
    context = re.sub(r"\s+", " ", text[snippet_start:snippet_end]).strip()
    period = "hour" if re.search(r"hour|/hr", context, re.I) else "year"

    return pd.Series(
        {
            "salary_min": float(match.group("minimum").replace(",", "")),
            "salary_max": float(match.group("maximum").replace(",", "")),
            "salary_currency": (
                match.group("currency") or SYMBOL_TO_CURRENCY[match.group("symbol")]
            ).upper(),
            "salary_period": period,
            "salary_snippet": context,
            "salary_source": "job_post",
        }
    )


salary_fields_df = jobs_df["description"].apply(extract_salary)
jobs_with_salary_df = pd.concat(
    [jobs_df.reset_index(drop=True), salary_fields_df.reset_index(drop=True)],
    axis=1,
)

salary_report_df = jobs_with_salary_df[
    jobs_with_salary_df["salary_min"].notna()
][
    [
        "job_id",
        "job_title",
        "location",
        "source_url",
        "salary_min",
        "salary_max",
        "salary_currency",
        "salary_period",
        "salary_snippet",
        "salary_source",
    ]
].sort_values(["salary_currency", "salary_min"], ascending=[True, False])

print(f"Total Waabi jobs: {len(jobs_df)}")
print(f"Jobs with salary ranges in the source post: {len(salary_report_df)}")
salary_report_df


Total Waabi jobs: 57
Jobs with salary ranges in the source post: 2


,job_id,job_title,location,source_url,salary_min,salary_max,salary_currency,salary_period,salary_snippet,salary_source
42,7fe1b29b-9f8b-4a51-8669-20724015ecd3,"Senior / Staff Software Engineer, Web Tools","Toronto, ON",https://jobs.lever.co/waabi/7fe1b29b-9f8b-4a51...,141000.0,249000.0,USD,year,m environment. The US yearly salary range for ...,job_post
54,bfe20894-5899-41b5-8cf7-c9f1469d18b7,"Systems Engineer, Platform Requirements and Ve...",Remote US,https://jobs.lever.co/waabi/bfe20894-5899-41b5...,140000.0,190000.0,USD,year,tical analysis The US yearly salary range for ...,job_post


## 4. Combine taxonomy and salary results

The merge uses `job_id`, which is retained directly from the scraper.
This keeps every derived field traceable to its original posting.


In [13]:
salary_merge_columns = [
    "job_id",
    "location",
    "salary_min",
    "salary_max",
    "salary_currency",
    "salary_period",
    "salary_snippet",
    "salary_source",
]

if classification_df.empty:
    final_report_df = pd.DataFrame(
        columns=classification_columns
        + [column for column in salary_merge_columns if column != "job_id"]
    )
    print("No classifications to combine yet. Set GROQ_API_KEY and rerun section 2.")
else:
    final_report_df = classification_df.merge(
        jobs_with_salary_df[salary_merge_columns],
        on="job_id",
        how="left",
        validate="one_to_one",
    )
    print(f"Combined rows: {len(final_report_df)}")

final_report_df


No classifications to combine yet. Set GROQ_API_KEY and rerun section 2.


,job_id,company,title,career_page_url,role_profile,skills,functional_area,location,salary_min,salary_max,salary_currency,salary_period,salary_snippet,salary_source


In [14]:
OUTPUT_PATH = PROJECT_ROOT / "data" / "waabi_classification_results.json"

if final_report_df.empty:
    print("Nothing exported because classification results are empty.")
else:
    final_report_df.to_json(
        OUTPUT_PATH,
        orient="records",
        indent=2,
        force_ascii=False,
    )
    print(f"Saved classification results to: {OUTPUT_PATH}")


Nothing exported because classification results are empty.


## 5. Conclusion and next steps

This notebook provides a reproducible Waabi classification prototype:

- scraper output is loaded from a versionable JSON file;
- required fields and duplicate IDs are validated before analysis;
- sample jobs are selected dynamically rather than by hard-coded IDs;
- LLM output is parsed and schema-checked before use;
- salary values come only from source advertisements;
- derived results retain job IDs and URLs for traceability.

Next, review the three sample classifications manually. Once the taxonomy
is accepted by the team, increase `SAMPLE_SIZE` gradually, add controlled
retry/rate-limit handling, and compare consistency across Waabi, Bosch,
Stack AV, and Waymo.
